# Correct Resolution Models

The following models take into account that, throughout analyses, *no_goal* seems to behave entirely differently to the other two Goal Type conditions (*goal_frequent* and *goal_non_frequent*). 

Previous focus structure effects (see notebooks 11 to 6) as well as further exhaustive modelling (see notebooks 15_appendix vs. 16_appendix) show consistently the heterogeneity of combined data; by spliting data into goal contexts and no-goal contexts, both effects as well as models improved despite the pruning of observations. No-goal contexts are also analysed separately; however, and although results are coherent with what is seen by comparing combined data to goal behaviour data, the sample is clearly not enough to show anything significant from no-goal contexts.

Taking into account the effects summary (notebook 5) and response opportunity analysis (notebook 6), these were chosen to be the best models to predict participants resolving succesfully.

Import Libraries

In [67]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Goal Behaviour Subgroup

1. Create resolved_correct (goals and no_goals)
2. Counts and Proportions (goals and no_goals)
3. Logistic Models

    - Goals

        Main Effect Models:

        - RC01: Focus

        Additive Models:

        - RC05: Focus + Goal_Type

        Interactive Models:

        - RC11: Focus * Agent + Goal_Type

    - No_goals

        Main Effect Models:

        - RC12: Focus
        - RC13: Agent

        Interactive Model:

        - RC15: Focus * Agent

4. Model Comparisons
5. Summary

Read Data

In [68]:
subgroup_theoretical = pd.read_csv("../../data/processed/subgroup_theoretical.csv")

1. Create resolved_correct

Binary Outcome: correct responses from escape_L2 observations.

In [69]:
subgroup_theoretical["escape_L2"] = (
    subgroup_theoretical["Response_Full"] != "L2_other"
).astype(int)

In [70]:
escapees = subgroup_theoretical[subgroup_theoretical["escape_L2"] == 1].copy()

In [71]:
escapees["resolved_correct"] = (escapees["Response_Full"] == "correct").astype(int)

Split escapees data into goals vs. no_goals

In [72]:
goals_escapees = escapees[escapees["Goal_Type"] != "no_goal"].copy()

In [73]:
no_goal_escapees = escapees[escapees["Goal_Type"] == "no_goal"].copy()

2. Counts and Proportions for goals and no_goals

General resolved_correct reponses count:

In [74]:
resolved_correct_counts = pd.crosstab(
    escapees["resolved_correct"],
    escapees["Response_Full"],
    margins= True)

resolved_correct_counts

Response_Full,L1_transfer,correct,missing_response,All
resolved_correct,,,,
0,96,0,3,99
1,0,61,0,61
All,96,61,3,160


resolved_correct responses count for goals:

In [75]:
goals_resolved_correct_counts = pd.crosstab(
    goals_escapees["resolved_correct"],
    goals_escapees["Response_Full"],
    margins= True)

goals_resolved_correct_counts

Response_Full,L1_transfer,correct,missing_response,All
resolved_correct,,,,
0,62,0,1,63
1,0,43,0,43
All,62,43,1,106


resolved_correct responses count for no_goals:

In [76]:
no_goal_resolved_correct_counts = pd.crosstab(
    no_goal_escapees["resolved_correct"],
    no_goal_escapees["Response_Full"],
    margins= True)

no_goal_resolved_correct_counts

Response_Full,L1_transfer,correct,missing_response,All
resolved_correct,,,,
0,34,0,2,36
1,0,18,0,18
All,34,18,2,54


General resolved_correct responses proportions:

In [77]:
resolved_correct_props = escapees["resolved_correct"].value_counts(normalize = True)
resolved_correct_props

resolved_correct
0    0.61875
1    0.38125
Name: proportion, dtype: float64

From 160 observations, almost 38% of responses that had escaped *L2_other* options were resolved correctly. 

resolved_correct responses proportions for goals:

In [78]:
goals_resolved_correct_props = goals_escapees["resolved_correct"].value_counts(normalize=True)
goals_resolved_correct_props

resolved_correct
0    0.59434
1    0.40566
Name: proportion, dtype: float64

From 106 observations, 40.5% of goal responses that had escaped *L2_other* options were resolved correctly. 

resolved_correct responses proportions for no_goals:

In [79]:
no_goal_resolved_correct_props = no_goal_escapees["resolved_correct"].value_counts(normalize=True)
no_goal_resolved_correct_props

resolved_correct
0    0.666667
1    0.333333
Name: proportion, dtype: float64

From 54 observations, 33% of no_goal responses that had escaped *L2_other* options were resolved correctly. 

- The proportion distribution of correct resolution responses is higher when there is a goal oriented representation (40.5%) as opposed to when there isn't a representation of a goal (33.3%). 

Counts and Proportions by Condition:

In [80]:
counts_condition = pd.crosstab(
    [escapees["Goal_Type"], escapees["Agent"],escapees["Focus"]],
    escapees["resolved_correct"],
    margins = True)

counts_condition

resolved_correct                0   1  All
Goal_Type         Agent Focus             
goal_frequent     0     I       7   9   16
                        They    7   5   12
                  1     I       7   7   14
                        They    5   3    8
goal_non_frequent 0     I       7   9   16
                        They   14   3   17
                  1     I      11   4   15
                        They    5   3    8
no_goal           0     I      15   6   21
                        They   10   3   13
                  1     I       9   5   14
                        They    2   4    6
All                            99  61  160

In [81]:
goals_counts_condition = pd.crosstab(
    [goals_escapees["Goal_Type"], goals_escapees["Agent"], goals_escapees["Focus"]],
    goals_escapees["resolved_correct"],
    margins = True)

goals_counts_condition

resolved_correct                0   1  All
Goal_Type         Agent Focus             
goal_frequent     0     I       7   9   16
                        They    7   5   12
                  1     I       7   7   14
                        They    5   3    8
goal_non_frequent 0     I       7   9   16
                        They   14   3   17
                  1     I      11   4   15
                        They    5   3    8
All                            63  43  106

In [82]:
no_goal_counts_condition = pd.crosstab(
    [no_goal_escapees["Goal_Type"], no_goal_escapees["Agent"], no_goal_escapees["Focus"]],
    no_goal_escapees["resolved_correct"],
    margins = True)

no_goal_counts_condition

resolved_correct        0   1  All
Goal_Type Agent Focus             
no_goal   0     I      15   6   21
                They   10   3   13
          1     I       9   5   14
                They    2   4    6
All                    36  18   54

In [83]:
props_condition = pd.crosstab(
    [escapees["Goal_Type"], escapees["Agent"], escapees["Focus"]],
    escapees["resolved_correct"],
    normalize= "index")

props_condition

resolved_correct                      0         1
Goal_Type         Agent Focus                    
goal_frequent     0     I      0.437500  0.562500
                        They   0.583333  0.416667
                  1     I      0.500000  0.500000
                        They   0.625000  0.375000
goal_non_frequent 0     I      0.437500  0.562500
                        They   0.823529  0.176471
                  1     I      0.733333  0.266667
                        They   0.625000  0.375000
no_goal           0     I      0.714286  0.285714
                        They   0.769231  0.230769
                  1     I      0.642857  0.357143
                        They   0.333333  0.666667

In [84]:
goals_props_condition = pd.crosstab(
    [goals_escapees["Goal_Type"], goals_escapees["Agent"], goals_escapees["Focus"]],
    goals_escapees["resolved_correct"],
    normalize= "index")

goals_props_condition

resolved_correct                      0         1
Goal_Type         Agent Focus                    
goal_frequent     0     I      0.437500  0.562500
                        They   0.583333  0.416667
                  1     I      0.500000  0.500000
                        They   0.625000  0.375000
goal_non_frequent 0     I      0.437500  0.562500
                        They   0.823529  0.176471
                  1     I      0.733333  0.266667
                        They   0.625000  0.375000

In [85]:
no_goal_props_condition = pd.crosstab(
    [no_goal_escapees["Goal_Type"], no_goal_escapees["Agent"], no_goal_escapees["Focus"]],
    no_goal_escapees["resolved_correct"],
    normalize= "index")

no_goal_props_condition

resolved_correct              0         1
Goal_Type Agent Focus                    
no_goal   0     I      0.714286  0.285714
                They   0.769231  0.230769
          1     I      0.642857  0.357143
                They   0.333333  0.666667

3. Logistic Models

Import Libraries

In [86]:
import statsmodels.formula.api as smf
from scipy.stats import chi2

Sanity checks:

In [87]:
escapees["resolved_correct"].value_counts()

resolved_correct
0    99
1    61
Name: count, dtype: int64

In [88]:
goals_escapees["resolved_correct"].value_counts()

resolved_correct
0    63
1    43
Name: count, dtype: int64

In [89]:
no_goal_escapees["resolved_correct"].value_counts()

resolved_correct
0    36
1    18
Name: count, dtype: int64

**Goal Context**

**Main Effect Models**

**RC01**

resolved_correct ~ Focus

In [90]:
RC01 = smf.logit(
    "resolved_correct ~ Focus",
    data=goals_escapees
    ).fit()

print(RC01.summary())

Optimization terminated successfully.
         Current function value: 0.661392
         Iterations 5
                           Logit Regression Results                           
Dep. Variable:       resolved_correct   No. Observations:                  106
Model:                          Logit   Df Residuals:                      104
Method:                           MLE   Df Model:                            1
Date:                Fri, 07 Aug 2026   Pseudo R-squ.:                 0.02051
Time:                        15:10:41   Log-Likelihood:                -70.108
converged:                       True   LL-Null:                       -71.575
Covariance Type:            nonrobust   LLR p-value:                   0.08664
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept        -0.0984      0.256     -0.384      0.701      -0.601       0.404
Focus[T.They]    -0.

Focus. again, gains more relevance. The model, however, is still not significant given that there are fewer observations.
This clearly differs from escaping L2 other options which was clearly driven by Agency (LLR p-value = 0.001628) and, as expected, shows a similar behaviour to resolving by means of transfer which was also siding towards Focus (LLR p-value= 0.06029) as opposed to Agency. 

**Additive Model**

**RC05**

resolved_correct ~ Focus + Goal_Type

In [91]:
RC05 = smf.logit(
    "resolved_correct ~ Focus + Goal_Type",
    data=goals_escapees
    ).fit()

print(RC05.summary())

Optimization terminated successfully.
         Current function value: 0.651957
         Iterations 5
                           Logit Regression Results                           
Dep. Variable:       resolved_correct   No. Observations:                  106
Model:                          Logit   Df Residuals:                      103
Method:                           MLE   Df Model:                            2
Date:                Fri, 07 Aug 2026   Pseudo R-squ.:                 0.03448
Time:                        15:10:41   Log-Likelihood:                -69.107
converged:                       True   LL-Null:                       -71.575
Covariance Type:            nonrobust   LLR p-value:                   0.08476
                                     coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept                          0.1894      0.329      0.576     

Focus alone's LLR p-value = 0.0866; together, LLR p-value = 0.0848. Approaching significance, not there yet, but slightly better than the model with Focus on its own.

**Interactive Model**

**RC11**

resolved_correct ~ Focus * Agent + Goal_Type

In [92]:
RC11 = smf.logit(
    "resolved_correct ~ Focus * Agent + Goal_Type",
    data=goals_escapees
    ).fit()

print(RC11.summary())

Optimization terminated successfully.
         Current function value: 0.640518
         Iterations 5
                           Logit Regression Results                           
Dep. Variable:       resolved_correct   No. Observations:                  106
Model:                          Logit   Df Residuals:                      101
Method:                           MLE   Df Model:                            4
Date:                Fri, 07 Aug 2026   Pseudo R-squ.:                 0.05142
Time:                        15:10:41   Log-Likelihood:                -67.895
converged:                       True   LL-Null:                       -71.575
Covariance Type:            nonrobust   LLR p-value:                    0.1180
                                     coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept                          0.5378      0.418      1.287     

This full model is run to be compared to transfer resolution (LLR p-value: 0.05318) Focus p-value reaches 0.031. The Goal type coefficient is very similar to the previous model RC05 with focus and goal type (-0.5703), so its contribution is steady even though it doesn't improve its p-value. The interaction doesn't reach significance. 

**No Goal Context**

**Main Effect Models**

**RC12**

resolved_correct ~ Focus 

In [93]:
RC12 = smf.logit(
    "resolved_correct ~ Focus",
    data=no_goal_escapees
    ).fit()

print(RC12.summary())

Optimization terminated successfully.
         Current function value: 0.635021
         Iterations 5
                           Logit Regression Results                           
Dep. Variable:       resolved_correct   No. Observations:                   54
Model:                          Logit   Df Residuals:                       52
Method:                           MLE   Df Model:                            1
Date:                Fri, 07 Aug 2026   Pseudo R-squ.:                0.002346
Time:                        15:10:41   Log-Likelihood:                -34.291
converged:                       True   LL-Null:                       -34.372
Covariance Type:            nonrobust   LLR p-value:                    0.6880
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept        -0.7802      0.364     -2.143      0.032      -1.494      -0.067
Focus[T.They]     0.

**RC13**

resolved_correct ~ Agent 

In [94]:
RC13 = smf.logit(
    "resolved_correct ~ Agent",
    data=no_goal_escapees
    ).fit()

print(RC13.summary())

Optimization terminated successfully.
         Current function value: 0.618743
         Iterations 5
                           Logit Regression Results                           
Dep. Variable:       resolved_correct   No. Observations:                   54
Model:                          Logit   Df Residuals:                       52
Method:                           MLE   Df Model:                            1
Date:                Fri, 07 Aug 2026   Pseudo R-squ.:                 0.02792
Time:                        15:10:41   Log-Likelihood:                -33.412
converged:                       True   LL-Null:                       -34.372
Covariance Type:            nonrobust   LLR p-value:                    0.1659
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -1.0217      0.389     -2.628      0.009      -1.784      -0.260
Agent          0.8210      0.

**Additive Model**

**Interactive Model**

**RC15**

resolved_correct ~ Focus * Agent 

In [95]:
RC15 = smf.logit(
    "resolved_correct ~ Focus * Agent",
    data=no_goal_escapees
    ).fit()

print(RC15.summary())

Optimization terminated successfully.
         Current function value: 0.602407
         Iterations 5
                           Logit Regression Results                           
Dep. Variable:       resolved_correct   No. Observations:                   54
Model:                          Logit   Df Residuals:                       50
Method:                           MLE   Df Model:                            3
Date:                Fri, 07 Aug 2026   Pseudo R-squ.:                 0.05358
Time:                        15:10:41   Log-Likelihood:                -32.530
converged:                       True   LL-Null:                       -34.372
Covariance Type:            nonrobust   LLR p-value:                    0.2977
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
Intercept              -0.9163      0.483     -1.897      0.058      -1.863       0.030
Fo

4. Model Comparisons

In [96]:
from scipy.stats import chi2

def lr_test(model_small, model_large):

    LR = 2 * (model_large.llf - model_small.llf)

    df = model_large.df_model - model_small.df_model

    p = chi2.sf(LR, df)

    return pd.Series({
        "LR": LR,
        "df": df,
        "p_value": p
    })

**Unique contributions to the additive model**

Focus (RC01)

↓

Focus + Goal_Type (RC05)


In [97]:
lr_test(RC01, RC05)

LR         2.000187
df         1.000000
p_value    0.157280
dtype: float64

- Does Goal_Type add information after Focus?

Not significantly again but better once the no-goal condition has been removed (p-value = 0.226). Goal contributes more cleanly.

**Interaction contributions**

Focus + Goal_Type (RC05)

↓

Focus * Agency + Goal_Type (RC11)

In [98]:
lr_test(RC05, RC11)

LR         2.425185
df         2.000000
p_value    0.297425
dtype: float64

- Does the Focus-Agent interaction improve the Focus + Goal Type additive account?

This interaction got worse than when data was combined (p-value = 0.086 vs. 0.157) once we removed the no_goals. 

**No_Goal contribution**

Agent (RC13)

↓

Focus * Agent (RC15)

In [99]:
lr_test(RC13, RC15)

LR         1.764250
df         2.000000
p_value    0.413902
dtype: float64

- Does their relationship require an interaction?

Now, compared to the no-goal escaping data (p-value = 0.91), turns out that the interaction while is still not required, it is much better than for escaping, suggesting different mechanisms for escaping vs. resolution for no-goals too.

5. Summary 

Overall:

- For escaping L2, Agency woke up and Focus helped once Agency was present.

- Once the data has been isolated by goal behaviour and considering both tranfer and succesful resolution, Focus becomes the dominant story, not Agency.  

- For correct resolution, Focus becomes stronger, Goal Type seems to contribute a little but Agency does not play a role. This reveals that there could be different cognitive processes involved in leaving the L2-other space compared to succesful resolution.

From model comparisons:

- Focus is the only predictor that keeps sitting on the edge of significance after Goal is controlled for correct resolution too once the data focuses on goal behaviour.

- The no-goal items on their own do not seem to merely introduce statistical noise; they appear to follow a qualitatively different behavioural strategy to goals. Across all exploratory analyses, no-goal items consistently displayed patterns that contrasted with those observed in goal-related items. This consistency across descriptive summaries, coefficient rankings, likelihood-ratio tests and model comparisons suggests that no-goal items may reflect a different behavioural regime rather than merely contributing random variability. Unfortunately, there aren't enough observations to make any claims.